# 🧠 Customer Churn Prediction: Feature Engineering, Model Training & SHAP Explainability (Days 4–9)

**Author:** Adarsh Kumar Pandey  
**Dataset:** data/telco_customer_churn_clean.csv  
**Algorithms:** Logistic Regression, Random Forest, XGBoost  
**Explainability Framework:** SHAP (SHapley Additive exPlanations)

## 1. Import Libraries & Set Styles

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
)
import shap

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

## 2. Load Cleaned Dataset & Feature Engineering

In [ ]:
df = pd.read_csv('../data/telco_customer_churn_clean.csv')
print('Data Shape:', df.shape)
df.head()

In [ ]:
# Feature Engineering: Tenure cohorts, add-on services, financial ratios & risk indicators
bins = [-1, 12, 24, 36, 48, 60, 72]
labels = ['0-12m', '13-24m', '25-36m', '37-48m', '49-60m', '61-72m']
df['tenure_cohort'] = pd.cut(df['tenure'], bins=bins, labels=labels)

service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in service_cols:
    df[f'{col}_flag'] = (df[col] == 'Yes').astype(int)

df['total_addon_services'] = df[[f'{col}_flag' for col in service_cols]].sum(axis=1)
df['security_bundle'] = ((df['OnlineSecurity'] == 'Yes') & (df['TechSupport'] == 'Yes')).astype(int)
df['streaming_bundle'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)

df['charge_ratio'] = df['TotalCharges'] / (df['tenure'] + 1.0)
df['monthly_to_total_ratio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1.0)
df['has_family'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).astype(int)

df['is_month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)
df['is_fiber_optic'] = (df['InternetService'] == 'Fiber optic').astype(int)
df['is_electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
df['high_monthly_charges'] = (df['MonthlyCharges'] >= 70.0).astype(int)

drop_cols = ['customerID'] + [f'{col}_flag' for col in service_cols]
df_ml = df.drop(columns=drop_cols)

y = (df_ml['Churn'] == 'Yes').astype(int)
X = df_ml.drop(columns=['Churn'])

cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print('Total engineered features:', X_encoded.shape[1])

## 3. Stratified Train / Test Split & Preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Train Models (Logistic Regression, Random Forest, XGBoost)

In [ ]:
scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'), X_train_scaled, X_test_scaled),
    'Random Forest': (RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_split=6, min_samples_leaf=3, random_state=42, class_weight='balanced', n_jobs=-1), X_train.values, X_test.values),
    'XGBoost': (XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42, n_jobs=-1), X_train.values, X_test.values)
}

fitted_models = {}
probs = {}
preds = {}

for name, (clf, X_tr, X_te) in models.items():
    clf.fit(X_tr, y_train)
    fitted_models[name] = clf
    preds[name] = clf.predict(X_te)
    probs[name] = clf.predict_proba(X_te)[:, 1]
    print(f'Done training: {name}')

## 5. Model Evaluation & ROC Curves

In [ ]:
metrics_list = []
for name in models.keys():
    metrics_list.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds[name]),
        'Balanced Accuracy': balanced_accuracy_score(y_test, preds[name]),
        'Precision': precision_score(y_test, preds[name]),
        'Recall': recall_score(y_test, preds[name]),
        'F1-Score': f1_score(y_test, preds[name]),
        'ROC-AUC': roc_auc_score(y_test, probs[name])
    })

pd.DataFrame(metrics_list).set_index('Model')

## 6. SHAP Explainability (TreeExplainer)

In [ ]:
explainer = shap.TreeExplainer(fitted_models['XGBoost'])
shap_values = explainer(X_test.values)

# Summary beeswarm plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values.values, X_test, feature_names=X_encoded.columns.tolist(), max_display=15)